# Part A — Exploratory Survival Analysis (Prepayment)

**(i)** Kaplan-Meier survival estimates for prepayment.
**(ii)** Implied hazard rates (discrete monthly + smoothed).
**(iii)** Stratified survival/hazard curves across FICO, LTV, DTI, vintage,
     original interest rate, loan purpose, and origination channel.
**(iv)** Robustness diagnostics: competing-risks CIF, bin-free Cox ranking,
     age × calendar heatmap.

All plots saved to `../figures/` as PNG for presentation use.

**On non-prepayment terminations.** Defaults and other terminations are treated
as right-censored observations for prepayment survival, which is the standard
single-event KM approach. This overstates cumulative prepayment incidence
because it implicitly assumes defaulted loans would have prepaid at the same
rate as survivors. The Aalen-Johansen section at the end shows the correct
competing-risks decomposition.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

import gc
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from lifelines.statistics import multivariate_logrank_test
from lifelines.utils import survival_table_from_events
from src.credit_data import load_loans, load_outcomes

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
print("repo:", REPO_ROOT)
print("figures →", FIGURES_DIR)


In [ ]:
YEARS = list(range(2006, 2023))   # in-sample vintages

# Numeric covariates — cut into labelled bins.
# Edit breaks/labels to change stratification; rest of notebook adapts.
COVARIATE_BINS = {
    "fico": {
        "breaks": [660, 700, 740, 780],
        "labels": ["<660", "660-700", "700-740", "740-780", "780+"],
    },
    "ltv": {
        "breaks": [60, 80, 90, 95],
        "labels": ["≤60", "60-80", "80-90", "90-95", "95+"],
    },
    "dti": {
        "breaks": [30, 40, 45],
        "labels": ["≤30", "30-40", "40-45", "45+"],
    },
    "orig_rate": {
        "breaks": [3.5, 4.5, 5.5, 6.5],
        "labels": ["<3.5%", "3.5-4.5%", "4.5-5.5%", "5.5-6.5%", "6.5%+"],
    },
}

# Categorical covariates — define as explicit Polars expression masks.
LOAN_PURPOSE_GROUPS = {
    "Purchase (P)":      pl.col("loan_purpose") == "P",
    "No-cash refi (N)":  pl.col("loan_purpose") == "N",
    "Cash-out refi (C)": pl.col("loan_purpose") == "C",
}
CHANNEL_GROUPS = {
    "Retail (R)":        pl.col("channel") == "R",
    "Broker (B)":        pl.col("channel") == "B",
    "Correspondent (T)": pl.col("channel") == "T",
    "TPO/Other (C)":     pl.col("channel") == "C",
}
VINTAGE_GROUPS = {
    "2006-2008 (crisis)":   [2006, 2007, 2008],
    "2009-2014 (recovery)": [2009, 2010, 2011, 2012, 2013, 2014],
    "2015-2019 (low-rate)": [2015, 2016, 2017, 2018, 2019],
    "2020-2022 (covid)":    [2020, 2021, 2022],
}

MAX_MONTHS      = 240    # upper horizon cap
SMOOTH_BW       = 3      # months, centered rolling-mean smoother window
SAMPLE_PER_GROUP = 500_000  # per-group cap for stratified plots (RAM)

# AT_RISK_FLOOR: the plot horizon is auto-truncated at the month where the
# at-risk pool first falls below this threshold. Set to 0 to disable.
AT_RISK_FLOOR = 100_000

# Initialised here; updated after aggregate KM is computed.
EFFECTIVE_MAX_MONTHS = MAX_MONTHS

# Accumulates one log-rank summary per plot_stratified() call.
LOGRANK_RESULTS: list[dict] = []


In [ ]:
loans = (
    load_loans(
        years=YEARS,
        columns=[
            "vintage_year",
            "fico", "ltv", "dti", "orig_rate",
            "loan_purpose", "channel",
            "event_type", "event_time_months",
        ],
    )
    .with_columns(
        (pl.col("event_type") == "prepaid").cast(pl.Int8).alias("prepay_observed"),
    )
    .filter(pl.col("event_time_months").is_not_null())
)
print(f"loans loaded: {loans.height:,} rows  vintages {YEARS[0]}-{YEARS[-1]}")
print("event_type distribution:")
print(loans.group_by("event_type").len().sort("len", descending=True))
print(f"\noverall prepayment rate: {loans['prepay_observed'].mean():.3f}")
loans = loans.drop("event_type")


## (i) Aggregate Kaplan-Meier survival for prepayment

S(t) = P(loan has not prepaid by month t).

The x-axis is truncated where the at-risk pool drops below `AT_RISK_FLOOR` loans —
below that count, the discrete hazard estimator is too noisy to be meaningful.


In [ ]:
dur = loans["event_time_months"].to_numpy()
evt = loans["prepay_observed"].to_numpy()

table_agg = survival_table_from_events(dur, evt)
table_agg = table_agg[table_agg.index <= MAX_MONTHS]
hazard_agg  = (table_agg["observed"] / table_agg["at_risk"]).rename("hazard")
S_agg       = (1.0 - hazard_agg).cumprod().rename("survival")

# Auto-truncate: find last month with enough at-risk loans.
if AT_RISK_FLOOR > 0:
    eligible = table_agg.index[table_agg["at_risk"] >= AT_RISK_FLOOR]
    EFFECTIVE_MAX_MONTHS = int(eligible.max()) if len(eligible) else MAX_MONTHS
else:
    EFFECTIVE_MAX_MONTHS = MAX_MONTHS
print(f"Effective horizon: {EFFECTIVE_MAX_MONTHS} months  "
      f"(at-risk ≥ {AT_RISK_FLOOR:,}  |  floor at {AT_RISK_FLOOR:,} loans)")

fig, ax = plt.subplots(figsize=(9, 5))
ax.step(S_agg.index, S_agg.values, where="post", lw=1.6,
        label=f"all loans (n={len(dur):,})")
ax.set_xlim(0, EFFECTIVE_MAX_MONTHS)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Months from origination")
ax.set_ylabel("P(not prepaid)")
ax.set_title(f"Kaplan-Meier: prepayment survival, vintages {YEARS[0]}-{YEARS[-1]}")
ax.grid(alpha=0.3); ax.legend()
fig.savefig(FIGURES_DIR / "partA_km_aggregate.png")
plt.show(); plt.close(fig)

for t in (12, 36, 60, 120, 180):
    avail = S_agg.index[S_agg.index <= t]
    if len(avail):
        print(f"S({t:>3} mo) ≈ {S_agg.loc[avail.max()]:.3f}")

del dur, evt
gc.collect()


## (ii) Implied hazard rates

Both panels are derived from the same lifetable. The right tail is truncated
at `EFFECTIVE_MAX_MONTHS` (the horizon where at-risk loans ≥ `AT_RISK_FLOOR`).
Beyond that point the discrete `events / at_risk` estimator is dominated by
sampling noise rather than the underlying prepayment rate.


In [ ]:
window = max(1, 2 * SMOOTH_BW + 1)
discrete_hazard = hazard_agg
smoothed = discrete_hazard.rolling(window=window, center=True, min_periods=1).mean()
at_risk  = table_agg["at_risk"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(discrete_hazard[:EFFECTIVE_MAX_MONTHS].index,
             discrete_hazard[:EFFECTIVE_MAX_MONTHS].values * 100, lw=1.0)
axes[0].set_xlabel("Months from origination")
axes[0].set_ylabel("Monthly prepay hazard (%)")
axes[0].set_title("Discrete monthly hazard")
axes[0].set_xlim(0, EFFECTIVE_MAX_MONTHS); axes[0].grid(alpha=0.3)

axes[1].plot(smoothed[:EFFECTIVE_MAX_MONTHS].index,
             smoothed[:EFFECTIVE_MAX_MONTHS].values * 100, lw=1.5, color="tab:orange")
axes[1].set_xlabel("Months from origination")
axes[1].set_ylabel("Smoothed monthly hazard (%/mo)")
axes[1].set_title(f"Rolling-mean smoother (window={window} mo)")
axes[1].set_xlim(0, EFFECTIVE_MAX_MONTHS); axes[1].grid(alpha=0.3)

fig.suptitle(f"Aggregate prepayment hazard, vintages {YEARS[0]}-{YEARS[-1]}", y=1.02)
fig.savefig(FIGURES_DIR / "partA_hazard_aggregate.png")
plt.show(); plt.close(fig)

print(f"peak hazard: {discrete_hazard.max()*100:.2f}% at month {discrete_hazard.idxmax()}")
print(f"at-risk at truncation point (mo {EFFECTIVE_MAX_MONTHS}): "
      f"{int(at_risk.loc[EFFECTIVE_MAX_MONTHS]):,}" if EFFECTIVE_MAX_MONTHS in at_risk.index
      else "")
gc.collect()


## (iii) Stratified survival and hazard by covariate

For each covariate the left panel shows KM survival curves and the right shows
the smoothed monthly hazard, both truncated at `EFFECTIVE_MAX_MONTHS`. The
log-rank p-values and chi-squared statistics are printed inline and collected
for the summary panel at the end.


In [ ]:
def _lifetable(d, e, max_m):
    t = survival_table_from_events(d, e)
    t = t[t.index <= max_m]
    h = (t["observed"] / t["at_risk"]).rename("hazard")
    s = (1.0 - h).cumprod().rename("survival")
    return h, s


def plot_stratified(loans_df, group_col, save_stem,
                    bin_cfg=None, explicit_groups=None,
                    title=None, smooth_bw=SMOOTH_BW,
                    max_months=None, sample_per_group=SAMPLE_PER_GROUP):
    """KM + smoothed hazard stratified by group_col; log-rank appended to LOGRANK_RESULTS."""
    if max_months is None:
        max_months = EFFECTIVE_MAX_MONTHS

    if bin_cfg is not None:
        df = (loans_df
              .filter(pl.col(group_col).is_not_null())
              .with_columns(pl.col(group_col)
                              .cut(breaks=bin_cfg["breaks"], labels=bin_cfg["labels"])
                              .alias("__grp")))
        ordered_groups = bin_cfg["labels"]
    elif explicit_groups is not None:
        cols = []
        ordered_groups = list(explicit_groups.keys())
        for label, mask in explicit_groups.items():
            cols.append(pl.when(mask).then(pl.lit(label)).otherwise(None))
        df = (loans_df
              .with_columns(pl.coalesce(cols).alias("__grp"))
              .filter(pl.col("__grp").is_not_null()))
    else:
        raise ValueError("provide bin_cfg or explicit_groups")

    if sample_per_group is not None:
        parts = []
        for grp in ordered_groups:
            sub = df.filter(pl.col("__grp") == grp)
            if sub.height > sample_per_group:
                sub = sub.sample(n=sample_per_group, seed=0)
            parts.append(sub)
        df = pl.concat(parts)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    cmap   = plt.cm.viridis(np.linspace(0.0, 0.9, len(ordered_groups)))
    window = max(1, 2 * smooth_bw + 1)

    for color, label in zip(cmap, ordered_groups):
        sub = df.filter(pl.col("__grp") == label)
        if sub.height < 100:
            continue
        d = sub["event_time_months"].to_numpy()
        e = sub["prepay_observed"].to_numpy()
        h, s = _lifetable(d, e, max_months)
        smooth = h.rolling(window=window, center=True, min_periods=1).mean()
        axes[0].step(s.index, s.values, where="post",
                     color=color, lw=1.4, label=f"{label} (n={sub.height:,})")
        axes[1].plot(smooth.index, smooth.values * 100,
                     color=color, lw=1.4, label=label)
        del sub, d, e, h, s, smooth
        gc.collect()

    axes[0].set_xlim(0, max_months); axes[0].set_ylim(0, 1.02)
    axes[0].set_xlabel("Months from origination")
    axes[0].set_ylabel("P(not prepaid)")
    axes[0].set_title(f"KM by {group_col}"); axes[0].grid(alpha=0.3); axes[0].legend()
    axes[1].set_xlim(0, max_months)
    axes[1].set_xlabel("Months from origination")
    axes[1].set_ylabel("Smoothed hazard (%/mo)")
    axes[1].set_title(f"Hazard by {group_col}"); axes[1].grid(alpha=0.3); axes[1].legend()
    fig.suptitle(title or f"Prepayment by {group_col}", y=1.02)
    fig.savefig(FIGURES_DIR / f"partA_strat_{save_stem}.png")
    plt.show(); plt.close(fig)

    pdf = (df.select(["event_time_months", "prepay_observed", "__grp"])
             .drop_nulls().to_pandas())
    res = multivariate_logrank_test(
        pdf["event_time_months"], pdf["__grp"], pdf["prepay_observed"])
    summary = {
        "covariate": save_stem,
        "n_groups":  int(pdf["__grp"].nunique()),
        "n_loans":   int(len(pdf)),
        "chi2":      float(res.test_statistic),
        "p_value":   float(res.p_value),
    }
    LOGRANK_RESULTS.append(summary)
    print(f"  log-rank: chi2={summary['chi2']:,.1f}  p={summary['p_value']:.2e}  "
          f"n_groups={summary['n_groups']}  n={summary['n_loans']:,}")
    del df, pdf, res, fig, axes
    gc.collect()
    return summary


### FICO

In [ ]:
plot_stratified(loans, "fico", save_stem="fico", bin_cfg=COVARIATE_BINS["fico"]);

### LTV

In [ ]:
plot_stratified(loans, "ltv", save_stem="ltv", bin_cfg=COVARIATE_BINS["ltv"]);

### DTI

In [ ]:
plot_stratified(loans, "dti", save_stem="dti", bin_cfg=COVARIATE_BINS["dti"]);

### Vintage group

In [ ]:
plot_stratified(loans, "vintage_year", save_stem="vintage",
                explicit_groups={lbl: pl.col("vintage_year").is_in(yrs) for lbl, yrs in VINTAGE_GROUPS.items()},
                title="Prepayment by vintage group");

### Original interest rate

In [ ]:
plot_stratified(loans, "orig_rate", save_stem="orig_rate", bin_cfg=COVARIATE_BINS["orig_rate"]);

### Loan purpose

In [ ]:
plot_stratified(loans, "loan_purpose", save_stem="loan_purpose",
                explicit_groups=LOAN_PURPOSE_GROUPS, title="Prepayment by loan purpose");

### Origination channel

In [ ]:
plot_stratified(loans, "channel", save_stem="channel",
                explicit_groups=CHANNEL_GROUPS, title="Prepayment by origination channel");

## Statistical comparison — log-rank tests

With n ≈ 13M the p-value is essentially zero for every covariate; what matters is
the relative magnitude of chi². Log-rank chi² scales linearly with n, so the bar
chart gives the *ranking* of each covariate's univariate power to separate survival
curves, not an absolute significance test.

Key caveat: vintage absorbs most of the rate-cycle variation (calendar time = refi
incentive = vintage × current rate). Its large chi² will shrink substantially once
time-varying rate incentive is included as a covariate in the Cox model (Part B).

Log-rank also assumes proportional hazards; covariates whose group curves cross
(e.g. high-LTV bursting early vs. low-LTV being more persistent) may have their
effect understated.


In [ ]:
results = (
    pd.DataFrame(LOGRANK_RESULTS)
    .sort_values("chi2", ascending=False)
    .reset_index(drop=True)
)
rd = results.copy()
rd["chi2"]    = rd["chi2"].round(1)
rd["p_value"] = rd["p_value"].apply(lambda p: f"{p:.2e}")
rd.to_csv(FIGURES_DIR / "partA_logrank_results.csv", index=False)
print("saved:", FIGURES_DIR / "partA_logrank_results.csv")
rd


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
order = results.sort_values("chi2")
ax.barh(order["covariate"], order["chi2"], color="tab:blue", alpha=0.8)
for i, (cov, chi2) in enumerate(zip(order["covariate"], order["chi2"])):
    ax.text(chi2, i, f"  {chi2:,.0f}", va="center", fontsize=9)
ax.set_xlabel("Log-rank chi-squared statistic")
ax.set_title("Strength of stratification of prepayment survival, by covariate")
ax.grid(axis="x", alpha=0.3)
fig.savefig(FIGURES_DIR / "partA_logrank_summary.png")
plt.show(); plt.close(fig)


## How to customise

- **Different bins** — edit `COVARIATE_BINS` (breaks + labels) or `*_GROUPS` dicts.
- **Different vintage groups** — edit `VINTAGE_GROUPS`.
- **Horizon** — `AT_RISK_FLOOR` drives auto-truncation; set to 0 to disable.
- **Smoother** — `SMOOTH_BW` months.
- **Vintages** — `YEARS` (add 2024-2025 for held-out eval).
- **RAM** — `SAMPLE_PER_GROUP` (default 500k; raise or set to None for full-precision).
- **New covariate** — call `plot_stratified(loans, "col", save_stem="col", bin_cfg=...)`.


---
## Diagnostics and robustness

The three sections below address methodological questions about the KM results above.


### Competing risks: Aalen-Johansen cumulative incidence functions

KM treats defaulted/other-terminated loans as censored observations, implicitly
assuming they would have prepayed at the same rate as survivors — an assumption
that is unlikely to hold (low-FICO, high-LTV borrowers who default would have been
slower prepayers). The **cumulative incidence function** (CIF) from the
Aalen-Johansen estimator is the correct quantity: the probability that a loan
terminates *due to cause k specifically* by month t, in the presence of
competing events.

CIF_prepay + CIF_default + CIF_other ≤ 1 at each t; the gap is the probability
of still being active. Compare 1 − S_KM (above) with CIF_prepay here: KM
overstates the cumulative prepayment probability to the extent that it ignores
the competitor (default).


In [ ]:
from lifelines import AalenJohansenFitter

# Load outcomes fresh — `loans` already dropped event_type.
aj_raw = (
    load_outcomes(years=YEARS, columns=["event_time_months", "event_type"])
    .filter(pl.col("event_time_months").is_not_null())
    .to_pandas()
)
# If still large, sample (AJ on 2M+ rows can be slow).
if len(aj_raw) > 1_000_000:
    aj_raw = aj_raw.sample(1_000_000, random_state=0)

event_map = {"censored": 0, "prepaid": 1, "defaulted": 2, "other_termination": 3}
aj_raw["event_code"] = aj_raw["event_type"].map(event_map).fillna(0).astype(int)

T = aj_raw["event_time_months"]
E = aj_raw["event_code"]

# Fit once — cumulative_density_ is a DataFrame, one column per event code.
ajf = AalenJohansenFitter(calculate_variance=False)
ajf.fit(T, E)
cif = ajf.cumulative_density_

fig, ax = plt.subplots(figsize=(9, 5))
cause_labels = {
    1: ("Prepaid (CIF)",           "tab:blue"),
    2: ("Defaulted (CIF)",         "tab:red"),
    3: ("Other termination (CIF)", "tab:orange"),
}
for cause, (label, color) in cause_labels.items():
    if cause in cif.columns:
        data = cif[cause][cif.index <= EFFECTIVE_MAX_MONTHS]
        ax.step(data.index, data.values, where="post", color=color, lw=1.5, label=label)

# Also plot 1 - KM for comparison
ax.step(S_agg[:EFFECTIVE_MAX_MONTHS].index,
        1 - S_agg[:EFFECTIVE_MAX_MONTHS].values,
        where="post", color="tab:blue", lw=1.2, ls="--",
        label="1 − KM survival (prepay, ignores competing risks)")

ax.set_xlim(0, EFFECTIVE_MAX_MONTHS); ax.set_ylim(0, 1.0)
ax.set_xlabel("Months from origination")
ax.set_ylabel("Cumulative incidence")
ax.set_title("Aalen-Johansen CIF vs KM\n(dashed = KM overstates prepay by ignoring defaults)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.savefig(FIGURES_DIR / "partA_competing_risks_cif.png")
plt.show(); plt.close(fig)

del aj_raw, T, E
gc.collect()


### Bin-free univariate Cox ranking

Log-rank test results depend on the choice of bin boundaries. Univariate
proportional-hazards models treat each numeric covariate *continuously*, giving a
ranking that is independent of binning assumptions. The likelihood-ratio chi² from
each univariate Cox fit is the proper bin-free analog of the log-rank statistic.

Interpretation: a larger LR chi² means stronger evidence that the covariate
has a non-zero association with prepayment timing, treating the effect as
log-linear in the covariate. The hazard ratio shows direction and magnitude.


In [ ]:
from lifelines import CoxPHFitter

NUMERIC_COVARIATES = ["fico", "ltv", "dti", "orig_rate"]
cox_rows = []

for col in NUMERIC_COVARIATES:
    sub = (
        loans.select(["event_time_months", "prepay_observed", col])
        .drop_nulls()
        .to_pandas()
    )
    if len(sub) > 500_000:
        sub = sub.sample(500_000, random_state=0)
    cph = CoxPHFitter().fit(sub, "event_time_months", "prepay_observed", formula=col)
    lrt = cph.log_likelihood_ratio_test()
    cox_rows.append({
        "covariate":   col,
        "n":           len(sub),
        "hazard_ratio": float(np.exp(cph.params_[col])),
        "cox_lr_chi2": float(lrt.test_statistic),
        "logrank_chi2": next((r["chi2"] for r in LOGRANK_RESULTS if r["covariate"]==col), None),
    })
    del sub
    gc.collect()

cox_df = pd.DataFrame(cox_rows).sort_values("cox_lr_chi2", ascending=False).reset_index(drop=True)
cox_df["cox_lr_chi2"]  = cox_df["cox_lr_chi2"].round(1)
cox_df["hazard_ratio"] = cox_df["hazard_ratio"].round(4)
cox_df


In [ ]:
# Side-by-side bar: log-rank chi² vs Cox LR chi² for the numeric covariates.
fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(cox_df))
w = 0.38
ax.barh(x - w/2, cox_df["cox_lr_chi2"],   w, label="Cox LR chi²", color="tab:blue",   alpha=0.8)
ax.barh(x + w/2, cox_df["logrank_chi2"],   w, label="Log-rank chi²", color="tab:orange", alpha=0.8)
ax.set_yticks(x); ax.set_yticklabels(cox_df["covariate"])
ax.set_xlabel("Chi-squared statistic")
ax.set_title("Bin-free Cox vs binned log-rank: numeric covariates\n"
             "(similar ranking = binning assumptions not driving results)")
ax.legend(); ax.grid(axis="x", alpha=0.3)
fig.savefig(FIGURES_DIR / "partA_cox_vs_logrank.png")
plt.show(); plt.close(fig)
gc.collect()


### Age × calendar-year heatmap

The humps in the aggregate hazard curve appear at different loan ages for
different vintages because the same calendar event (e.g. 2020 rate drop) hits
each vintage at a different seasoning stage. This heatmap shows the monthly
prepayment rate as a function of **both** calendar year (rows) and loan age
(columns) simultaneously.

Reading the chart:
- **Horizontal bands** (same colour across ages within one year) = pure period
  effect (rate environment). A 2020 horizontal band well above neighbouring years
  means every age group prepays faster in 2020.
- **Vertical bands** (same colour across years at one age) = pure seasoning/burnout
  effect. A bright column at 36-48 months would mean loans of that age always have
  elevated hazard regardless of the rate environment.
- In practice you see both — the Cox model in Part B decomposes them by including
  time-varying rate incentive as a covariate.

Computed from the outcomes table only — no monthly panel scan required.


In [ ]:
hm = (
    load_outcomes(years=YEARS, columns=["vintage_year", "last_obs_date", "event_type"])
    .filter(pl.col("last_obs_date").is_not_null())
    .with_columns(pl.col("last_obs_date").dt.year().alias("end_year"))
)

# Pre-aggregate to (vintage_year, end_year, event_type) — ~1 600 rows.
summary_pl = (
    hm.group_by(["vintage_year", "end_year", "event_type"])
    .agg(pl.len().alias("n"))
)
sm = summary_pl.to_pandas()
active_sm  = sm.groupby(["vintage_year", "end_year"])["n"].sum().reset_index()
prepaid_sm = sm[sm["event_type"] == "prepaid"][["vintage_year", "end_year", "n"]]

CAL_YEARS  = list(range(2007, 2023))   # calendar years
AGE_YEARS  = list(range(0, 17))        # 0-16 years from origination
MIN_ACTIVE = 2_000                     # sparse cells set to NaN

matrix = np.full((len(CAL_YEARS), len(AGE_YEARS)), np.nan)
for ci, cy in enumerate(CAL_YEARS):
    for ai, ay in enumerate(AGE_YEARS):
        vy = cy - ay   # vintage year for this age × calendar combination
        if vy < YEARS[0] or vy > YEARS[-1]:
            continue
        n_act = int(active_sm[
            (active_sm.vintage_year == vy) & (active_sm.end_year >= cy)
        ]["n"].sum())
        if n_act < MIN_ACTIVE:
            continue
        n_pre = int(prepaid_sm[
            (prepaid_sm.vintage_year == vy) & (prepaid_sm.end_year == cy)
        ]["n"].sum())
        matrix[ci, ai] = n_pre / n_act / 12 * 100   # monthly rate

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(
    matrix,
    aspect="auto",
    origin="lower",
    cmap="YlOrRd",
    vmin=0,
    vmax=np.nanpercentile(matrix, 95),   # cap at 95th pct to avoid outlier wash-out
)
ax.set_xticks(range(len(AGE_YEARS)));  ax.set_xticklabels(AGE_YEARS)
ax.set_yticks(range(len(CAL_YEARS)));  ax.set_yticklabels(CAL_YEARS)
ax.set_xlabel("Loan age (years from origination)")
ax.set_ylabel("Calendar year")
ax.set_title("Monthly prepayment rate (%) — age × calendar year\n"
             "Horizontal banding = macro/rate effect; Vertical = seasoning/burnout")
plt.colorbar(im, ax=ax, label="Monthly prepay rate (%)")
fig.savefig(FIGURES_DIR / "partA_heatmap_age_calendar.png")
plt.show(); plt.close(fig)

del hm, summary_pl, sm, active_sm, prepaid_sm, matrix
gc.collect()


## Saved artifacts (for the presentation)

```
figures/
├── partA_km_aggregate.png
├── partA_hazard_aggregate.png
├── partA_strat_fico.png
├── partA_strat_ltv.png
├── partA_strat_dti.png
├── partA_strat_vintage.png
├── partA_strat_orig_rate.png        ← new
├── partA_strat_loan_purpose.png     ← new
├── partA_strat_channel.png          ← new
├── partA_logrank_summary.png
├── partA_logrank_results.csv
├── partA_competing_risks_cif.png    ← new
├── partA_cox_vs_logrank.png         ← new
└── partA_heatmap_age_calendar.png   ← new
```
